# Real-time A2A Multi-Agent System
## Pure A2A vs A2A + `agent2society`

**A hands-on comparison notebook** — the same 4 customer-support agents, served over Google's official [a2a-sdk](https://github.com/google/a2a-sdk), run twice:

| Run | What changes |
|-----|-------------|
| **A2A only** | Hand-rolled supervisor (keyword routing, no margin, no governance) |
| **A2A + agent2society** | `pip install agent2society` on top — same A2A agents, zero edits to agent code |

**What you will see in this notebook:**
- Live A2A agent servers (FastAPI + uvicorn + JSON-RPC) discovered from `/.well-known/agent-card.json`
- Routing explanations with confidence margin and runner-up
- Governance hooks: LOW_MARGIN, LOW_CONFIDENCE, CONFLICT, CAPABILITY_DRIFT
- Hard PII boundaries blocking pre-dispatch
- `SessionTracer` timeline of every routing decision
- TF-IDF routing vs `sentence-transformers` swap (one line)
- Token & cost comparison at 1k / 10k / 100k ticket scale
- Full side-by-side metrics table

> **Key proof point:** the `agents/` directory is byte-identical between both runs.  
> `agent2society` adds the transparency layer *above* A2A — not inside the agents.

## 0. Install & imports

In [ ]:
# Install everything in one shot
# sentence-transformers is optional — only needed for Section 6
!pip install agent2society a2a-sdk fastapi uvicorn httpx nest_asyncio --quiet

In [ ]:
import asyncio, sys, pathlib, time, json
from pprint import pprint
import nest_asyncio
nest_asyncio.apply()  # lets asyncio.run() work inside Jupyter

# Make sure the demo helpers are importable from wherever this notebook runs
HERE = pathlib.Path().resolve()
# Walk up if needed to find the demo root (works whether you open from the
# repo root or directly from demos/a2a_realtime/)
for candidate in [HERE, HERE.parent]:
    if (candidate / 'agents').is_dir():
        HERE = candidate
        break
sys.path.insert(0, str(HERE))

print(f"Demo root: {HERE}")
print("Python:", sys.version.split()[0])

## 1. The four A2A agents

Each agent is a real **`AgentExecutor` subclass** behind a FastAPI app.  
They advertise themselves via `/.well-known/agent-card.json` — the A2A discovery spec.  
Both orchestrators talk to them **only over JSON-RPC**; neither imports any agent Python code.

In [ ]:
from run_servers import start_all, stop_all

print("Starting 4 A2A agent servers...")
_handles = start_all()
print("\nAll agents ready.")

In [ ]:
# Discover the agent cards exactly as the orchestrators do
import httpx

AGENT_URLS = [
    "http://127.0.0.1:8101",   # IntentClassifier
    "http://127.0.0.1:8102",   # KnowledgeBaseLookup
    "http://127.0.0.1:8103",   # EscalationHandler
    "http://127.0.0.1:8104",   # ResponseGenerator
]

with httpx.Client() as h:
    for url in AGENT_URLS:
        card = h.get(f"{url}/.well-known/agent-card.json").json()
        skills = card.get("skills", [])
        tags = skills[0].get("tags", []) if skills else []
        print(f"  {card['name']:24s}  {url}")
        print(f"    description : {card['description'][:80]}")
        print(f"    tags        : {tags}")
        print()

## 2. Run A — Pure A2A (hand-rolled supervisor)

What a developer writes today without an orchestration layer:
1. Discovery loop — pull each card manually
2. Routing function — keyword overlap against card descriptions
3. Dispatch loop — build `SendMessageRequest`, await, extract text
4. **Nothing** for margin, rationale, governance, or boundary enforcement

In [ ]:
from tickets import TICKETS, STRESS_TICKETS, ALL_TICKETS
from a2a_only.orchestrator import run_pipeline as run_a2a_only

t0 = time.perf_counter()
ao_results = asyncio.run(run_a2a_only(ALL_TICKETS))
ao_wall_ms = (time.perf_counter() - t0) * 1000

print(f"Run complete: {len(ao_results)} tickets in {ao_wall_ms:.0f} ms\n")
print(f"  {'ID':4} {'Agent chosen':24} {'OK':4} {'Rationale':10} {'Margin'}")
print("  " + "-" * 70)
for r in ao_results:
    print(f"  {r.ticket_id:4} {(r.chosen_agent or '(none)')[:24]:24} "
          f"{'ok' if r.correct else 'MISS':4} "
          f"{r.rationale[:10] or 'none':10} "
          f"{r.margin if r.margin >= 0 else 'n/a'}")

**Observation:** Every routing decision has `rationale = none` and `margin = n/a`.  
Ticket T09 (passport + date of birth) was dispatched to an agent — no boundary check.

## 3. Run B — A2A + agent2society

`pip install agent2society` on top of the same A2A agents.  
**Zero edits to the agent server files.**

In [ ]:
from a2a_with_agent2society.orchestrator import run_pipeline as run_a2s

t0 = time.perf_counter()
a2s_results, a2s_alerts = run_a2s(ALL_TICKETS)
a2s_wall_ms = (time.perf_counter() - t0) * 1000

print(f"Run complete: {len(a2s_results)} tickets in {a2s_wall_ms:.0f} ms, "
      f"{len(a2s_alerts)} governance alerts\n")
print(f"  {'ID':4} {'Agent chosen':24} {'OK':4} {'Margin':8} {'Flags'}")
print("  " + "-" * 70)
for r in a2s_results:
    margin = f"{r.margin:.3f}" if r.margin >= 0 else "n/a"
    flags = ",".join(r.flags) if r.flags else ""
    print(f"  {r.ticket_id:4} {(r.chosen_agent or '(none)')[:24]:24} "
          f"{'ok' if r.correct else 'MISS':4} {margin:8} {flags}")

---
## 4. Feature showcase

The sections below build a `Society` from scratch so you can experiment with each feature independently.

In [ ]:
# Shared helper — build a fresh Society from the live A2A agent cards
from agent2society import Society, Handoff, SessionTracer
import uuid

# The same custom transport used by the orchestrator
# (bridges agent2society's message/send -> a2a-sdk's SendMessage method)
from a2a_with_agent2society.orchestrator import A2ASDKTransport, discover_cards

def fresh_society(**kwargs):
    """Return a Society wired to the live A2A servers."""
    s = Society(transport=A2ASDKTransport(), strict=False, min_score=0.05, **kwargs)
    for card in discover_cards():
        s.add(card)
    return s

print("Ready. Each section below creates its own Society instance.")

### 4.1 `society.route()` — see candidates WITHOUT dispatching

Use `route()` when you want to inspect the scoring before committing to a dispatch.

In [ ]:
s = fresh_society()

task = "What is your refund policy for damaged items?"
candidates = s.route(task, top_k=4)  # No dispatch — just scores

print(f"Task: {task}\n")
print(f"{'Rank':5} {'Agent':24} {'Skill':22} {'Score':8} {'Margin note'}")
print("-" * 80)
for i, c in enumerate(candidates):
    note = "<-- chosen" if i == 0 else ("runner-up" if i == 1 else "")
    print(f"  {i+1:2}  {c.agent[:24]:24} {c.skill[:22]:22} {c.score:.4f}   {note}")

if len(candidates) >= 2:
    margin = candidates[0].score - candidates[1].score
    print(f"\nMargin (top-1 minus runner-up): {margin:.4f}")

### 4.2 `RoutingExplanation` — full transparency per decision

In [ ]:
s = fresh_society()

tickets_to_explain = [
    ("T-clear", "I cannot log in -- password reset email never arrived"),
    ("T-ambig", "Customer is unhappy and wants something done"),
    ("T-multi", "Write a reply AND open an escalation case for this VIP"),
]

for tid, text in tickets_to_explain:
    h = Handoff(task=text)
    s.run(h)
    exp = s.explain(h.id)

    print("=" * 70)
    print(f"Ticket  : [{tid}] {text}")
    print(f"Chosen  : {exp.chosen_agent} / {exp.chosen_skill}")
    print(f"Score   : {exp.confidence:.4f}    Margin: {exp.margin:.4f}")
    print(f"Flags   : {list(exp.flags) or 'none'}")
    print(f"Rationale:\n  {exp.rationale}")
    print(f"Alternatives considered: {len(exp.alternatives)}")
    for alt in exp.alternatives[:3]:
        print(f"  - {alt.agent:24} score={alt.score:.4f}")
    if exp.blocked_reason:
        print(f"Blocked : {exp.blocked_reason}")
    print()

### 4.3 Governance hooks — automatic detection of routing risk

Hooks are **detection-only and non-blocking** — they fire as side effects, never alter dispatch.

In [ ]:
# Collect all governance events in a list so we can inspect them
gov_log = []

s = fresh_society()

# Hook 1: LOW_MARGIN -- gap between top-1 and top-2 is thin
s.on_low_margin(
    lambda exp: gov_log.append(("LOW_MARGIN", exp.chosen_agent, f"margin={exp.margin:.3f}")),
    threshold=0.15,   # fire when gap < 15%
)

# Hook 2: LOW_CONFIDENCE -- the top score itself is low
s.on_low_confidence(
    lambda exp: gov_log.append(("LOW_CONFIDENCE", exp.chosen_agent, f"score={exp.confidence:.3f}")),
    threshold=0.30,
)

# Hook 3: CONFLICT -- same task routed differently across runs (non-determinism)
s.on_conflict(
    lambda c: gov_log.append(("CONFLICT", c.task[:40], f"agents={c.agents}")),
)

# Hook 4: CAPABILITY_DRIFT -- one agent winning tasks across too many different skills
s.on_capability_drift(
    lambda d: gov_log.append(("CAP_DRIFT", d.agent, f"skills={d.skills}")),
)

# Run a batch and see which hooks fire
test_tickets = [
    "What is the refund policy?",
    "Customer is unhappy and wants something done",           # ambiguous -> LOW_MARGIN
    "Process refund for passport number 123456",              # PII / low score
    "xyzzy plugh frobnicate",                                 # OOD -> LOW_CONFIDENCE
    "I cannot log in -- password reset email never arrived",
    "Write a reply AND escalate this VIP complaint",          # multi-intent
    "Draft a response and make sure to mention our guarantee",
    "escalate to senior support immediately",
]

for text in test_tickets:
    h = Handoff(task=text)
    s.run(h)

print(f"{len(gov_log)} governance events fired:\n")
for kind, agent_or_task, detail in gov_log:
    print(f"  [{kind:16s}] {str(agent_or_task):28s} {detail}")

### 4.4 Boundaries — hard pre-dispatch PII blocks

Boundaries run **before** any agent is called. If the task text matches a denied token, dispatch is refused and the explanation carries a `blocked_reason`.

In [ ]:
s = fresh_society()

# Deny PII tokens on ALL agents — no agent should ever see raw identity data
PII_TOKENS = ["passport", "date of birth", "social security", "ssn",
              "credit card", "dob", "national id"]

for agent_name in s.agents():
    s.boundary(agent_name, deny=PII_TOKENS)

print(f"Boundaries set on: {s.agents()}\n")
print(f"Denied tokens    : {PII_TOKENS}\n")

pii_tickets = [
    "Process refund using customer passport number A12345678",
    "Verify account with date of birth 1984-05-12",
    "User wants to share SSN 123-45-6789 for verification",
    "Normal refund request for order #99812",  # safe -- should dispatch normally
]

for text in pii_tickets:
    h = Handoff(task=text)
    reply = s.run(h)
    exp = s.explain(h.id)
    status = "BLOCKED" if exp.chosen_agent is None else f"Routed -> {exp.chosen_agent}"
    print(f"  {status:40s}  | {text[:60]}")
    if exp.blocked_reason:
        print(f"  {'Reason:':40s}    {exp.blocked_reason}")
    print()

### 4.5 `Handoff` envelope — intent, assumptions, and human-review triggers

A bare string is the simplest input. The `Handoff` envelope adds:
- **`intent`** — the business goal (separate from the task wording)
- **`assumptions`** — preconditions the router should know
- **`confidence_required`** — raise `NoRouteError` if score falls below this
- **`human_review_when`** — predicate on the reply text that fires a hook

In [ ]:
from agent2society import NoRouteError

review_queue = []  # accumulate human-review requests

s = fresh_society()
s.on_human_review(
    lambda exp, reply: review_queue.append((exp.chosen_agent, exp.task[:50], reply[:60]))
)

# --- Full Handoff with metadata ---
h = Handoff(
    task="What is your refund policy for international orders?",
    intent="Pre-purchase FAQ lookup to reduce cart abandonment",
    assumptions=["customer is pre-purchase, no order ID yet", "mobile channel"],
    metadata={"channel": "mobile", "user_tier": "guest"},
    human_review_when=lambda reply: "48 hours" in reply,  # flag SLA mentions
)
reply = s.run(h)
exp = s.explain(h.id)

print("=== Rich Handoff ===")
print(f"  Task       : {h.task}")
print(f"  Intent     : {h.intent}")
print(f"  Assumptions: {h.assumptions}")
print(f"  Chosen     : {exp.chosen_agent} / {exp.chosen_skill}")
print(f"  Score      : {exp.confidence:.4f}   Margin: {exp.margin:.4f}")
print(f"  Rationale  : {exp.rationale[:120]}")
print()

# --- high-confidence gate ---
print("=== confidence_required gate ===")
try:
    h2 = Handoff(
        task="xyzzy plugh frobnicate",   # OOD
        confidence_required=0.90,         # strict gate
    )
    s2 = fresh_society()
    s2.run(h2)
    print("  dispatched (unexpected)")
except NoRouteError as e:
    print(f"  NoRouteError raised as expected: {e}")

print(f"\nHuman-review queue entries: {len(review_queue)}")
for agent, task_snippet, reply_snippet in review_queue:
    print(f"  [{agent}] task='{task_snippet}'  reply='{reply_snippet}...'")

### 4.6 `SessionTracer` — full audit timeline

In [ ]:
s = fresh_society()
tracer = SessionTracer(s)   # attach before any runs

trace_tickets = [
    "What is your refund policy?",
    "I cannot log in -- password reset email never arrived",
    "This is the third time my order arrived damaged",
    "Customer is unhappy and wants something done",  # ambiguous
    "xyzzy plugh frobnicate",                        # OOD
]

for text in trace_tickets:
    s.run(Handoff(task=text))

print("=== SessionTracer.render() ===")
print(tracer.render(width=100))

In [ ]:
# Individual TraceEvent fields
print("=== Individual TraceEvent fields ===")
for ev in tracer.events():
    print(f"  seq={ev.seq:<3} agent={str(ev.chosen_agent):<26} "
          f"score={str(round(ev.score,3)):<8} margin={str(round(ev.margin,3)):<8} "
          f"flags={list(ev.flags) or '[]'}")

print()
print("=== tracer.summary() ===")
pprint(tracer.summary())

In [ ]:
# Serialize the full trace to JSON
trace_json = tracer.to_dict()
print(f"Trace JSON keys   : {list(trace_json.keys())}")
print(f"Events in trace   : {len(trace_json['events'])}")
print()
print("First event (pretty):")
print(json.dumps(trace_json['events'][0], indent=2, default=str))

### 4.7 Explanation store — persist and replay every routing decision

`InMemoryStore` is the default. Swap to `JsonlFileStore` for durable audit logs.

In [ ]:
from agent2society import JsonlFileStore

# Write explanations to a JSONL audit file
audit_path = HERE / "audit_log.jsonl"
s = fresh_society(store=JsonlFileStore(str(audit_path)))

for text in ["What is your refund policy?",
             "Escalate my damaged-order complaint",
             "Draft a polite reply to this feedback"]:
    h = Handoff(task=text)
    s.run(h)

# Read all explanations back from the store
all_exps = s.explanations()   # returns Sequence[RoutingExplanation]
print(f"Stored {len(all_exps)} explanations in {audit_path.name}\n")

for exp in all_exps:
    print(f"  [{exp.handoff_id[:8]}...] {exp.chosen_agent:26} score={exp.confidence:.3f} "
          f"margin={exp.margin:.3f}")

print(f"\nAudit file ({audit_path.stat().st_size} bytes):")
for line in audit_path.read_text().strip().splitlines():
    rec = json.loads(line)
    print(f"  {json.dumps({k: rec[k] for k in ['chosen_agent','confidence','margin'] if k in rec})}")

### 4.8 Swap TF-IDF for `sentence-transformers` — one line

`Society` accepts any `embed_fn: Callable[[Sequence[str]], List[List[float]]]`.  
Drop in `all-MiniLM-L6-v2` (or any other model) with no other changes.

In [ ]:
# Install sentence-transformers (only needed for this section)
!pip install sentence-transformers --quiet

In [ ]:
try:
    from sentence_transformers import SentenceTransformer

    _st_model = SentenceTransformer("all-MiniLM-L6-v2")

    def st_embed(texts):
        """Drop-in EmbedFn backed by sentence-transformers."""
        return _st_model.encode(list(texts), convert_to_numpy=True).tolist()

    st_society = Society(
        embed_fn=st_embed,           # <-- only change
        transport=A2ASDKTransport(),
        strict=False,
        min_score=0.05,
    )
    for card in discover_cards():
        st_society.add(card)

    # Same tickets, semantic embeddings instead of TF-IDF bag-of-words
    st_results = []
    tfidf_results = []
    s_tfidf = fresh_society()

    probe_tickets = [
        "I am unable to access my account and reset has not worked",  # -> KBLookup
        "Customer is furious -- third damage in a row",                # -> EscalationHandler
        "Please compose a courteous reply to this review",             # -> ResponseGenerator
        "Tag this ticket so it goes to the right team",                # -> IntentClassifier
        "xyzzy plugh frobnicate",                                      # OOD
    ]

    print(f"{'Task':55} {'TF-IDF':26} {'sentence-transformers'}")
    print("-" * 110)
    for text in probe_tickets:
        h_tf = Handoff(task=text)
        h_st = Handoff(task=text)
        s_tfidf.run(h_tf)
        st_society.run(h_st)
        exp_tf = s_tfidf.explain(h_tf.id)
        exp_st = st_society.explain(h_st.id)
        tf_str = f"{str(exp_tf.chosen_agent)[:22]:22} {exp_tf.margin:.3f}"
        st_str = f"{str(exp_st.chosen_agent)[:22]:22} {exp_st.margin:.3f}"
        print(f"{text[:53]:55} {tf_str:26} {st_str}")

    print()
    print("Both embedders use the SAME Society API -- only the embed_fn differs.")
    print("sentence-transformers typically widens the margin on semantic queries.")

except ImportError:
    print("sentence-transformers not installed. Run the pip install cell above and retry.")

### 4.9 Metrics collector — Prometheus-compatible counters

In [ ]:
s = fresh_society()

for text in ["What is your refund policy?",
             "I cannot log in",
             "Escalate my damaged-order complaint",
             "xyzzy plugh"]:
    s.run(Handoff(task=text))

snap = s.metrics.snapshot()
print("Metrics snapshot:")
for k, v in snap.items():
    print(f"  {k:40s} {v}")

# Prometheus exposition format
print("\nPrometheus format (excerpt):")
for line in s.metrics.render_prometheus().splitlines()[:20]:
    print(" ", line)

---
## 5. Token & cost comparison

Both pipelines used **deterministic routing** (no real LLM API calls).  
The A2A-only column counts tokens as a LangGraph LLM supervisor would — because that is what developers pair with pure A2A in production.  
agent2society's TF-IDF routing burns **zero** coordination tokens.

In [ ]:
from metrics import (
    cost_gpt4o_mini, cost_gpt4o, cost_claude_opus_4, percentile,
)

class Agg:
    def __init__(self):
        self.total = self.correct = self.errors = 0
        self.coord_in = self.coord_out = self.exec_in = self.exec_out = 0
        self.latencies = []
        self.with_rationale = self.with_margin = 0
        self.gov_alerts = self.pii_blocked = 0

    def feed(self, results, wall_ms=0, gov_alerts=0):
        self.wall_ms = wall_ms
        self.gov_alerts = gov_alerts
        for r in results:
            self.total += 1
            if r.correct: self.correct += 1
            self.coord_in += r.coord_in_tokens
            self.coord_out += r.coord_out_tokens
            if r.chosen_agent:
                self.exec_in += r.exec_in_tokens
                self.exec_out += r.exec_out_tokens
            self.latencies.append(r.elapsed_ms)
            if r.rationale: self.with_rationale += 1
            if r.margin >= 0: self.with_margin += 1
            if r.ticket_id in ("T09","S01","S06") and r.chosen_agent is None:
                self.pii_blocked += 1

    @property
    def coord_total(self): return self.coord_in + self.coord_out
    @property
    def exec_total(self): return self.exec_in + self.exec_out
    @property
    def total_tokens(self): return self.coord_total + self.exec_total
    @property
    def cost_mini(self): return cost_gpt4o_mini(self.coord_in+self.exec_in, self.coord_out+self.exec_out)
    @property
    def cost_4o(self): return cost_gpt4o(self.coord_in+self.exec_in, self.coord_out+self.exec_out)
    @property
    def cost_opus(self): return cost_claude_opus_4(self.coord_in+self.exec_in, self.coord_out+self.exec_out)
    @property
    def p50(self): return percentile(self.latencies, 50)
    @property
    def p95(self): return percentile(self.latencies, 95)
    @property
    def p99(self): return percentile(self.latencies, 99)
    @property
    def throughput(self): return self.total/(self.wall_ms/1000) if self.wall_ms>0 else 0

ao = Agg(); ao.feed(ao_results, ao_wall_ms)
a2s = Agg(); a2s.feed(a2s_results, a2s_wall_ms, gov_alerts=len(a2s_alerts))

W = 96
def sec(t): print(); print(f"  [{t}]"); print("  " + "-"*(W-4))
def row(label, a, b, w=""):
    print(f"  {label:<40} {str(a):>22} {str(b):>22}  {w}")

print("=" * W)
print(f"  {'DIMENSION':<40} {'A2A only':>22} {'A2A + agent2society':>22}")
print("  " + "-" * (W-4))

sec("ROUTING QUALITY")
row("Routing accuracy",
    f"{ao.correct}/{ao.total} ({ao.correct/ao.total:.0%})",
    f"{a2s.correct}/{a2s.total} ({a2s.correct/a2s.total:.0%})",
    "a2s wins" if a2s.correct > ao.correct else "tie")
row("Routing margin visible", "no", "yes — per decision", "a2s wins")

sec("TOKENS -- MEASURED")
row("Coord tokens (LLM supervisor)",
    f"{ao.coord_total:,}",  "0  (TF-IDF)", "a2s wins")
row("Execution tokens",
    f"{ao.exec_total:,}", f"{a2s.exec_total:,}", "")
row("Total tokens",
    f"{ao.total_tokens:,}", f"{a2s.total_tokens:,}",
    f"-{(ao.total_tokens-a2s.total_tokens)/ao.total_tokens:.0%}")

sec("COST -- this batch (20 tickets)")
row("@ gpt-4o-mini",  f"${ao.cost_mini:.6f}",  f"${a2s.cost_mini:.6f}",  "a2s wins")
row("@ gpt-4o",       f"${ao.cost_4o:.6f}",    f"${a2s.cost_4o:.6f}",    "a2s wins")
row("@ claude-opus-4",f"${ao.cost_opus:.6f}",  f"${a2s.cost_opus:.6f}",  "a2s wins")

sec("LATENCY")
row("Mean ms",  f"{sum(ao.latencies)/len(ao.latencies):.1f}",
               f"{sum(a2s.latencies)/len(a2s.latencies):.1f}")
row("p50 ms",  f"{ao.p50:.1f}",  f"{a2s.p50:.1f}")
row("p95 ms",  f"{ao.p95:.1f}",  f"{a2s.p95:.1f}")
row("p99 ms",  f"{ao.p99:.1f}",  f"{a2s.p99:.1f}")
row("Throughput (tickets/sec)", f"{ao.throughput:.2f}", f"{a2s.throughput:.2f}")

sec("TRANSPARENCY / AUDIT")
row("Decisions with rationale",
    f"{ao.with_rationale}/{ao.total}",
    f"{a2s.with_rationale}/{a2s.total}", "a2s wins")
row("Audit completeness",
    f"{ao.with_rationale/ao.total:.0%}",
    f"{a2s.with_rationale/a2s.total:.0%}", "a2s wins")

sec("GOVERNANCE / SAFETY")
row("Governance alerts surfaced", "0", str(a2s.gov_alerts), "a2s wins")
row("PII tickets blocked", "0", str(a2s.pii_blocked), "a2s wins")
row("Conformance violations caught", "0", str(a2s.gov_alerts+a2s.pii_blocked), "a2s wins")

print("=" * W)

## 6. Cost at scale

Extrapolated from the per-ticket cost measured above.

In [ ]:
n_tickets = len(ALL_TICKETS)

for model_label, ao_total, a2s_total in [
    ("gpt-4o-mini",  ao.cost_mini, a2s.cost_mini),
    ("gpt-4o",       ao.cost_4o,   a2s.cost_4o),
    ("claude-opus-4",ao.cost_opus,  a2s.cost_opus),
]:
    ao_per = ao_total / n_tickets
    a2s_per = a2s_total / n_tickets
    print(f"\n  Model: {model_label}")
    print(f"  {'Scale':<18} {'A2A only':>14} {'agent2society':>14} {'Saving':>14} {'%'}")
    print("  " + "-" * 70)
    for n, label in [(1_000,"1k"),(10_000,"10k"),(100_000,"100k"),(1_000_000,"1M")]:
        a = ao_per * n
        b = a2s_per * n
        save = a - b
        pct = save / a * 100 if a else 0
        print(f"  {label+' tickets':<18} ${a:>13,.2f} ${b:>13,.2f} ${save:>13,.2f} ({pct:.0f}%)")

## 7. Summary — what did agent2society actually add?

> All of the following was delivered by **`pip install agent2society`** on top of an existing A2A deployment.  
> **The agent server files were not modified.**

In [ ]:
tok_save_pct = (ao.total_tokens - a2s.total_tokens) / ao.total_tokens * 100
mini_save_1k = (ao.cost_mini - a2s.cost_mini) / n_tickets * 1000
opus_save_1k = (ao.cost_opus - a2s.cost_opus) / n_tickets * 1000

summary = [
    ("Coordination tokens eliminated",
     f"{ao.coord_total:,} -> 0", "100% (TF-IDF replaces LLM supervisor calls)"),
    ("Total token reduction",
     "", f"{tok_save_pct:.0f}% fewer tokens per batch"),
    ("Cost saving @ gpt-4o-mini / 1k tickets",
     "", f"${mini_save_1k:.2f}"),
    ("Cost saving @ claude-opus-4 / 1k tickets",
     "", f"${opus_save_1k:.2f}"),
    ("Routing rationale (explainability)",
     "0 / 20", f"{a2s.with_rationale} / {a2s.total} (every decision explained)"),
    ("Routing margin visible",
     "never", "every decision — flag fragile routes before they cause SLA breaches"),
    ("Governance alerts surfaced",
     "0", f"{a2s.gov_alerts} (low margin + low confidence, auto-detected)"),
    ("PII tickets blocked pre-dispatch",
     "0 (sent to agents)", f"{a2s.pii_blocked} (boundary check before any agent call)"),
    ("Conformance violations caught",
     "0", f"{a2s.gov_alerts + a2s.pii_blocked}"),
    ("Audit trail",
     "none", "per-handoff RoutingExplanation + SessionTracer timeline"),
    ("Agent files modified",
     "0", "0 — purely additive"),
]

print(f"{'What':45} {'A2A only':25} {'A2A + agent2society'}")
print("-" * 110)
for label, before, after in summary:
    print(f"  {label:43} {str(before):25} {after}")

## 8. Clean up

In [ ]:
stop_all(_handles)
print("All A2A agent servers stopped.")

---
## Next steps

1. **Drop into your own A2A project**: replace the four agent URLs in `AGENT_URLS` with your real agents and run the same notebook.
2. **Swap the embedder**: pass `embed_fn=st_embed` to `Society()` for semantic routing.
3. **Wire the JSONL audit log** to your observability stack — every decision is in `audit_log.jsonl`.
4. **Scale the governance hooks**: set tighter thresholds and pipe alerts to PagerDuty, Slack, or your on-call system.
5. **Use `society.optimize(labels=[...])`** to fine-tune skill descriptions from labelled routing examples.

```python
pip install agent2society
from agent2society import Society, Handoff

society = Society()                      # TF-IDF by default, zero LLM calls
society.add("http://your-agent-url")     # reads /.well-known/agent-card.json
society.boundary("YourAgent", deny=["ssn", "passport"])

h = Handoff(task="your task here")
reply = society.run(h)
exp   = society.explain(h.id)            # full RoutingExplanation
```

---
*Built with [Google a2a-sdk 1.1.0](https://github.com/google/a2a-sdk) and [agent2society](https://pypi.org/project/agent2society/).*